# Text Generation with LLaDA-8B-Instruct

## Trustworthy AI — Watermarking Diffusion Language Models

This notebook reproduces the text generation procedure using
LLaDA-8B-Instruct and the watermarking implementation proposed
in *Watermarking Diffusion Language Models* (Gloaguen et al., 2026).

### Experimental setup

- Model: GSAI-ML/LLaDA-8B-Instruct
- Watermark: DiffusionKGW_Optimal_Gaussian (Ours)
- Datasets: Longform QA, Finance QA, AlpacaFarm (WaterBench)
- Samples: 200 prompts per dataset
- Conditions: Watermarked and Unwatermarked
- Total: 1,200 generated responses

### Generation strategy

Each dataset is divided into 10 batches of 20 prompts.

For readability, this notebook demonstrates the procedure using
batches 01–05 of Longform QA. The same procedure was repeated
for all remaining batches and datasets.

Results are stored as JSONL files for subsequent watermark
detection, robustness evaluation, and attack experiments.

### Reference

https://github.com/eth-sri/diffusion-lm-watermark

## 1. Environment Setup

The experiments are executed on Google Colab using an NVIDIA A100 GPU.

We use Python 3.12 and install the dependencies provided by
the official implementation.

The repository is cloned into the Colab runtime, and a dedicated
Python virtual environment is created using `uv`.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path("/content/diffusion-lm-watermark")
VENV = Path("/content/.venv")
PYTHON = VENV / "bin/python"

# Create a Python 3.12 virtual environment
if not PYTHON.exists():
    subprocess.run(
        ["uv", "venv", str(VENV), "--python", "3.12", "--seed"],
        check=True
    )

# Clone the official repository
if not (REPO / ".git").exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/eth-sri/diffusion-lm-watermark.git",
            str(REPO)
        ],
        check=True
    )

# Install dependencies
subprocess.run(
    [
        "uv", "pip", "install",
        "-r", str(REPO / "requirements.txt"),
        "--python", str(PYTHON),
        "--torch-backend=auto"
    ],
    check=True
)

# Install the watermarking package
subprocess.run(
    [
        "uv", "pip", "install",
        "-e", str(REPO),
        "--python", str(PYTHON)
    ],
    check=True
)

print("Environment setup completed.")

## 2. Environment Verification

Before generating text, we verify that:

1. CUDA is available.
2. The GPU is correctly detected.
3. The official WaterBench datasets are accessible.
4. The repository revision is recorded for reproducibility.

In [ ]:
import subprocess
from pathlib import Path

REPO = Path("/content/diffusion-lm-watermark")
PYTHON = Path("/content/.venv/bin/python")

# GPU and PyTorch verification
subprocess.run(
    [
        str(PYTHON), "-c",
        "import torch; "
        "print('CUDA available:', torch.cuda.is_available()); "
        "print('GPU:', torch.cuda.get_device_name(0) "
        "if torch.cuda.is_available() else 'None')"
    ],
    check=True
)

# Record the repository revision
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True
).strip()

print("Repository commit:", commit)

# Dataset verification
datasets = [
    "2-1_longform_qa",
    "2-2_finance_qa",
    "5-1_alpacafarm"
]

for dataset in datasets:
    path = REPO / "data/WaterBench" / f"{dataset}.jsonl"
    assert path.is_file(), f"Missing dataset: {path}"
    print(f"Dataset available: {dataset}")

## 3. Dataset Preparation

We select the first 200 prompts from each WaterBench dataset.

Each dataset is divided into 10 batches containing 20 prompts.

This batching strategy allows us to generate and save results
incrementally, reducing the risk of data loss if a Colab session
is interrupted.

For this demonstration, we prepare batches 01–05 of Longform QA.

The original prompt records are preserved without modification.

In [ ]:
import json
from pathlib import Path

REPO = Path("/content/diffusion-lm-watermark")

DATASET = "2-1_longform_qa"
NUM_PROMPTS = 200
BATCH_SIZE = 20
BATCH_IDS = range(1, 6)

SOURCE = REPO / "data/WaterBench" / f"{DATASET}.jsonl"
BATCH_DIR = Path("/content/watermark_batches")

BATCH_DIR.mkdir(parents=True, exist_ok=True)

# Read the first 200 prompts
with SOURCE.open(encoding="utf-8") as f:
    prompts = [
        json.loads(line)
        for line in f
        if line.strip()
    ][:NUM_PROMPTS]

assert len(prompts) == NUM_PROMPTS
assert all("input" in row and "context" in row for row in prompts)

# Create 5 batches of 20 prompts
for batch_id in BATCH_IDS:
    start = (batch_id - 1) * BATCH_SIZE
    end = batch_id * BATCH_SIZE

    selected = prompts[start:end]

    batch_path = (
        BATCH_DIR /
        f"{DATASET}_batch_{batch_id:02d}.jsonl"
    )

    with batch_path.open("w", encoding="utf-8") as f:
        for row in selected:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(
        f"{batch_path.name}: {len(selected)} prompts"
    )

print("Dataset preparation completed.")

## 4. Model and Watermark Configuration

We use LLaDA-8B-Instruct with the generation parameters
specified in the official implementation.

### Model parameters

| Parameter | Value |
|---|---|
| Model | GSAI-ML/LLaDA-8B-Instruct |
| Steps | 300 |
| Generation length | 300 |
| Temperature | 0.5 |
| Remasking | random |
| Block length | 25 |
| CFG scale | 0.0 |
| Precision | bfloat16 |

### Watermark parameters

| Parameter | Value |
|---|---|
| Watermark type | Ours |
| Delta | 4.0 |
| Gamma | 0.25 |
| Top-k | 50 |
| Seeding scheme | sumhash |
| Greenlist type | bernoulli |
| Iterations | 1 |

We evaluate two experimental conditions:

1. Watermarked generation using the proposed watermark.
2. Unwatermarked generation using the same model and
   generation parameters.

The watermark strength delta = 4.0 corresponds to our
experimental configuration.

In [ ]:
import yaml
from pathlib import Path

DATASET = "2-1_longform_qa"

BATCH_DIR = Path("/content/watermark_batches")
CONFIG_DIR = Path("/content/production_configs")

CONFIG_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CONFIG = {
    "model_name": "GSAI-ML/LLaDA-8B-Instruct",
    "tokenizer_name": "GSAI-ML/LLaDA-8B-Instruct",
    "torch_dtype": "bfloat16",
    "device_map": "cuda",
    "trust_remote_code": True,
    "steps": 300,
    "gen_length": 300,
    "temperature": 0.5,
    "remasking": "random",
    "model_specific_arguments": {
        "cfg_scale": 0.0,
        "mask_id": 126336,
        "block_length": 25
    }
}

WM_CONFIG = {
    "watermark_type": "DiffusionKGW_Optimal_Gaussian",
    "delta": 4.0,
    "convolution_kernel": [-1],
    "topk": 50,
    "seeding_scheme": "sumhash",
    "greenlist_type": "bernoulli",
    "greenlist_params": {
        "gamma": 0.25
    },
    "enforce_kl": False,
    "n_iter": 1
}

NO_WM_CONFIG = {
    "watermark_type": "None"
}

PROMPT_TEMPLATES = {
    "2-1_longform_qa": (
        "You are a helpful assistant, please answer the "
        "following question within 300 words:\n"
        "{context}\n{input}"
    ),
    "2-2_finance_qa": (
        "You are a helpful assistant, please answer the "
        "following question with financial knowledge "
        "within 300 words:\n{context}\n{input}"
    ),
    "5-1_alpacafarm": (
        "You are a helpful assistant, please answer "
        "the following instruction: \n{context}\n{input}"
    )
}

for batch_id in range(1, 6):
    batch_name = f"{DATASET}_batch_{batch_id:02d}"
    batch_path = BATCH_DIR / f"{batch_name}.jsonl"

    assert batch_path.is_file()

    for condition in ["unwatermarked", "watermarked"]:

        watermark_enabled = condition == "watermarked"

        config = {
            "model_configuration": MODEL_CONFIG,
            "watermark_type": (
                "Ours" if watermark_enabled else "None"
            ),
            "watermark_config": (
                WM_CONFIG if watermark_enabled else NO_WM_CONFIG
            ),
            "evaluation_config": {
                "num_samples": 20,
                "batch_size": 1,
                "evaluation_datasets": [{
                    "path": str(batch_path),
                    "user_prompt": PROMPT_TEMPLATES[DATASET],
                    "chat": True,
                    "max_length": 150,
                    "min_length": 1
                }],
                "tqdm": True
            }
        }

        filename = f"{batch_name}_{condition}.yaml"
        config_path = CONFIG_DIR / filename

        config_path.write_text(
            yaml.safe_dump(config, sort_keys=False),
            encoding="utf-8"
        )

        print(f"Created: {filename}")

print("Configuration files created.")

## 5. Text Generation

We generate responses using the official `run_config.py`
script from the watermarking repository.

For each batch, the model produces 20 responses under
each experimental condition.

### Execution procedure

1. Load the corresponding YAML configuration.
2. Generate 20 completions with LLaDA-8B-Instruct.
3. Save the outputs in JSONL format.
4. Verify the number of responses and unique prompts.
5. Provide a download link for local storage.

Each batch uses a separate output directory to avoid
mixing results from different executions.

Existing result files are never overwritten automatically.

In [ ]:
import subprocess
import time
import json
import base64
import html

from pathlib import Path
from IPython.display import HTML, display

REPO = Path("/content/diffusion-lm-watermark")
PYTHON = Path("/content/.venv/bin/python")

CONFIG_DIR = Path("/content/production_configs")
RESULT_DIR = Path("/content/production_results")

DATASET = "2-1_longform_qa"
BATCH_IDS = range(1, 6)
CONDITIONS = ["unwatermarked", "watermarked"]

RESULT_DIR.mkdir(parents=True, exist_ok=True)

def download_link(source, filename):
    encoded = base64.b64encode(source.read_bytes()).decode("ascii")
    safe_name = html.escape(filename, quote=True)

    display(HTML(
        f'<a download="{safe_name}" '
        f'href="data:application/octet-stream;base64,{encoded}">'
        f'Download {safe_name}</a>'
    ))

for condition in CONDITIONS:

    print(f"\nCondition: {condition}", flush=True)

    for batch_id in BATCH_IDS:

        name = (
            f"{DATASET}_batch_{batch_id:02d}_{condition}"
        )

        config = CONFIG_DIR / f"{name}.yaml"
        output = RESULT_DIR / name
        result_file = output / "results.json"

        print(f"\n{'='*60}", flush=True)
        print(f"Batch {batch_id:02d}/05: {name}", flush=True)

        assert config.is_file(), f"Missing config: {config}"

        # Never overwrite existing results
        if result_file.exists():
            print("Existing results found. Skipping generation.")

        else:
            if output.exists():
                raise RuntimeError(
                    f"Incomplete output directory: {output}. "
                    "Inspect it before restarting."
                )

            output.mkdir(parents=True)

            start = time.time()

            process = subprocess.run(
                [
                    str(PYTHON),
                    "-u",
                    "scripts/run_config.py",
                    "--config", str(config),
                    "--output_path", str(output),
                    "--num_samples", "20"
                ],
                cwd=REPO
            )

            if process.returncode != 0:
                raise RuntimeError(
                    f"Generation failed: {name}. "
                    "Do not rerun before inspecting the output."
                )

            print(
                f"Generation time: {(time.time()-start)/60:.1f} min",
                flush=True
            )

        # Validate the generated file
        assert result_file.is_file(), (
            f"Missing result file: {result_file}"
        )

        with result_file.open(encoding="utf-8") as f:
            rows = [
                json.loads(line)
                for line in f
                if line.strip()
            ]

        n_rows = len(rows)
        n_unique = len({row["prompt"] for row in rows})

        print(f"Responses: {n_rows}")
        print(f"Unique prompts: {n_unique}/20")

        valid = (
            n_rows == 20
            and n_unique == 20
            and all(
                str(row.get("completion", "")).strip()
                for row in rows
            )
        )

        filename = (
            f"{name}.jsonl"
            if valid
            else f"{name}_RAW.jsonl"
        )

        # Download link after each batch
        download_link(result_file, filename)

        if not valid:
            raise RuntimeError(
                f"Unexpected results in {name}. "
                "Raw results preserved for manual inspection."
            )

        print(f"Batch {batch_id:02d} completed.", flush=True)

print("\nText generation completed.")

## 6. Results Validation

After generation, we verify the following properties:

- Exactly 20 responses per batch
- Exactly 20 unique prompts per batch
- No empty completions
- Valid JSONL formatting
- Matching prompts between the two experimental conditions

Unexpected files are preserved for manual inspection.

The original generation outputs are kept unchanged.

In [ ]:
import json
from pathlib import Path

RESULT_DIR = Path("/content/production_results")

DATASET = "2-1_longform_qa"
CONDITIONS = ["unwatermarked", "watermarked"]

all_valid = True

for batch_id in range(1, 6):

    prompts_by_condition = {}

    for condition in CONDITIONS:

        name = f"{DATASET}_batch_{batch_id:02d}_{condition}"
        path = RESULT_DIR / name / "results.json"

        if not path.is_file():
            print(f"MISSING: {name}")
            all_valid = False
            continue

        with path.open(encoding="utf-8") as f:
            rows = [
                json.loads(line)
                for line in f
                if line.strip()
            ]

        prompts = {row["prompt"] for row in rows}
        nonempty = all(
            str(row.get("completion", "")).strip()
            for row in rows
        )

        valid = (
            len(rows) == 20
            and len(prompts) == 20
            and nonempty
        )

        prompts_by_condition[condition] = prompts

        print(
            f"{name}: "
            f"{len(rows)} responses, "
            f"{len(prompts)} unique prompts, "
            f"valid={valid}"
        )

        all_valid = all_valid and valid

    if len(prompts_by_condition) == 2:
        matching = (
            prompts_by_condition["watermarked"]
            == prompts_by_condition["unwatermarked"]
        )

        print(f"Batch {batch_id:02d} matching prompts: {matching}")
        all_valid = all_valid and matching

print("\nAll checks passed:", all_valid)

## 7. Results Storage and Reproducibility

Google Colab uses temporary storage. Files saved under `/content/`
may disappear when the runtime is terminated.

All generated JSONL files must therefore be transferred to
persistent local storage before disconnecting the GPU.

The complete experiment consists of:

- 3 WaterBench datasets
- 200 prompts per dataset
- 2 generation conditions
- 1,200 generated responses

The remaining batches were generated using the same procedure,
with dataset-specific prompt templates.

Subsequent analysis is performed locally, without requiring
the generation GPU.